# E0: Graph Neural Network (TransformerConv + Set2Set) for DNA Thermodynamics (Standardized)

**Thesis:** Inductive Biases in Representation Learning for DNA Thermodynamic Property Prediction  
**Experiment ID:** E0  
**Thesis Chapter:** Chapter 2 — Baselines  

## Core Idea
DNA sequences are naturally graphs: nucleotides are **nodes**, and the backbone + hydrogen bonds are **edges**. This GNN encodes the molecule as:
- **Nodes:** 4-dimensional one-hot nucleotide identity
- **Edges:** 3-dimensional type (5′→3′, 3′→5′, H-bond)
- **Convolutions:** `TransformerConv` — graph-adapted self-attention
- **Pooling:** `Set2Set` — LSTM-based, permutation-invariant graph-level aggregation

**Inductive bias:** **Permutation equivariance** (order of nodes doesn't matter) + explicit relational structure (backbone vs H-bond edges are different).

In [1]:
# ── 1. Setup & Imports ────────────────────────────────────────────────────────
import os, json, time
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch_geometric.data import Data, InMemoryDataset
from torch_geometric.loader import DataLoader as PyGLoader
from torch_geometric.nn import TransformerConv, Set2Set
from torch.nn import ModuleList, Linear

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

import wandb
import sys
if sys.platform == 'win32' and not os.environ.get('WANDB_MODE'):
    os.environ['WANDB_MODE'] = 'online'

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available(): print(f'GPU: {torch.cuda.get_device_name(0)}')

COLORS = {'E0_GNN':'#7f8c8d','E1_1DCNN':'#3498db','E2_2DCNN':'#e74c3c',
          'E3_SAT':'#9b59b6','E4_PINN':'#e67e22','E5_Hybrid':'#1abc9c'}
MODEL_COLOR = COLORS['E0_GNN']

c:\Users\anant\anaconda3\envs\nnn_win\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda:0
GPU: NVIDIA GeForce GTX 1660 Ti


In [2]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
DATA_CSV   = 'data/models/raw/combined_dataset.csv'
SPLIT_JSON = 'data/models/raw/combined_data_split.json'
PROCESSED_DIR = 'data/models/processed_v2'   # separate from existing processed dir

config = dict(
    model_name      = 'GNN_TransformerConv_Set2Set',
    experiment_id   = 'E0',
    hidden_channels = 125,
    n_conv_layers   = 4,
    linear_hidden   = 128,
    conv_dropout    = 0.0127,
    linear_dropout  = 0.25,
    set2set_steps   = 10,
    n_epoch         = 200,
    batch_size      = 256,
    lr              = 1e-3,
    weight_decay    = 1e-5,
    grad_clip       = 1.0,
    dataset         = 'arr',
    wandb_project   = 'NNN_Thesis_Experiments',
    checkpoint_dir  = 'MyExperiments/GNN/models',
)
print('Config:', config)

Config: {'model_name': 'GNN_TransformerConv_Set2Set', 'experiment_id': 'E0', 'hidden_channels': 125, 'n_conv_layers': 4, 'linear_hidden': 128, 'conv_dropout': 0.0127, 'linear_dropout': 0.25, 'set2set_steps': 10, 'n_epoch': 200, 'batch_size': 256, 'lr': 0.001, 'weight_decay': 1e-05, 'grad_clip': 1.0, 'dataset': 'arr', 'wandb_project': 'NNN_Thesis_Experiments', 'checkpoint_dir': 'MyExperiments/GNN/models'}


In [3]:
# ── 3. Data Loading & Normalization ───────────────────────────────────────────
df = pd.read_csv(DATA_CSV, index_col='SEQID')
df.sort_index(inplace=True)
with open(SPLIT_JSON) as f:
    split = json.load(f)

# Train & evaluate on 'arr' only — lit_uv / ov are held-out generalization sets
TRAIN_DATASET = 'arr'

train_df = df.loc[split['train_ind']].dropna(subset=['dH','Tm'])
train_df = train_df[train_df['dataset'] == TRAIN_DATASET]
val_df   = df.loc[split['val_ind']  ].dropna(subset=['dH','Tm'])
val_df   = val_df[val_df['dataset'] == TRAIN_DATASET]
test_df  = df.loc[split['test_ind'] ].dropna(subset=['dH','Tm'])
test_df  = test_df[test_df['dataset'] == TRAIN_DATASET]

sumstats = {
    'dH_min': float(train_df['dH'].min()), 'dH_max': float(train_df['dH'].max()),
    'Tm_min': float(train_df['Tm'].min()), 'Tm_max': float(train_df['Tm'].max()),
}
def normalize(v, mn, mx):   return (v - mn) / (mx - mn)
def unnormalize(v, mn, mx): return v * (mx - mn) + mn

print(f'Train {len(train_df):,}  Val {len(val_df):,}  Test {len(test_df):,}  (arr only)')
print(f'dH [{sumstats["dH_min"]:.1f}, {sumstats["dH_max"]:.1f}]  Tm [{sumstats["Tm_min"]:.1f}, {sumstats["Tm_max"]:.1f}]')

Train 25,025  Val 1,318  Test 1,387  (arr only)
dH [-68.2, -2.7]  Tm [13.6, 68.6]


In [4]:
# ── 4. Graph Encoding ─────────────────────────────────────────────────────────

def onehot_nucleotide(seq):
    """Returns (N, 4) array — one-hot over {A,T,C,G}."""
    m = {'A':0,'T':1,'C':2,'G':3}
    arr = np.zeros((len(seq),4), dtype=np.float32)
    for i,nt in enumerate(seq.upper()):
        if nt in m: arr[i,m[nt]] = 1.
    return arr


def dotbracket_to_edges(struct):
    """
    Converts dot-bracket to edge_index (2, E) and edge_attr (E, 3).
    Edge types: [is_5to3, is_3to5, is_hbond]
    Uses a stack for correct nested H-bond matching.
    """
    clean = struct.replace('+','')
    N = len(clean)
    strand_break = struct.find('+')

    if strand_break == -1:
        backbone = [[i, i+1] for i in range(N-1)]
    else:
        # skip bond across the '+' separator
        backbone = [[i, i+1] for i in range(N-1) if i != strand_break-1]

    # Stack-based parenthesis matching for H-bonds
    stack, hbonds = [], []
    for i, ch in enumerate(clean):
        if   ch == '(': stack.append(i)
        elif ch == ')' and stack: hbonds.append([stack.pop(), i])

    edges = backbone + [e[::-1] for e in backbone] + hbonds + [e[::-1] for e in hbonds]
    n_bb, n_hb = len(backbone), len(hbonds)
    attr = np.zeros((len(edges), 3), dtype=np.float32)
    attr[:n_bb, 0] = 1.           # 5'→3'
    attr[n_bb:2*n_bb, 1] = 1.     # 3'→5'
    attr[2*n_bb:, 2] = 1.         # H-bond
    if not edges:
        return torch.zeros((2,0),dtype=torch.long), torch.zeros((0,3),dtype=torch.float)
    return (torch.tensor(np.array(edges).T, dtype=torch.long),
            torch.tensor(attr, dtype=torch.float))


def row_to_graph(row, sumstats):
    refseq = str(row['RefSeq'])
    if '[' in refseq:
        try: refseq = ''.join(eval(refseq))
        except: pass
    struct = str(row['TargetStruct'])
    x = torch.tensor(onehot_nucleotide(refseq.replace('+','')), dtype=torch.float)
    ei, ea = dotbracket_to_edges(struct)
    dH_n = normalize(row['dH'], sumstats['dH_min'], sumstats['dH_max'])
    Tm_n = normalize(row['Tm'], sumstats['Tm_min'], sumstats['Tm_max'])
    return Data(x=x, edge_index=ei, edge_attr=ea, y=torch.tensor([dH_n,Tm_n],dtype=torch.float))

# Quick sanity check
_g = row_to_graph(df.iloc[0], sumstats)
print(f'Sample graph — nodes: {_g.x.shape}  edges: {_g.edge_index.shape}  y: {_g.y}')

Sample graph — nodes: torch.Size([16, 4])  edges: torch.Size([2, 42])  y: tensor([0.5923, 0.7044])


In [5]:
# ── 5. Graph Dataset & DataLoaders ────────────────────────────────────────────

class NNNGraphDataset(InMemoryDataset):
    """
    Builds and caches a PyG InMemoryDataset from combined_dataset.csv.
    Saves to PROCESSED_DIR to avoid colliding with the original processed files.
    """
    def __init__(self, df, sumstats, root, transform=None):
        self._df = df
        self._ss = sumstats
        super().__init__(root, transform)
        self.data, self.slices = torch.load(self.processed_paths[0])

    @property
    def raw_file_names(self): return []

    @property
    def processed_file_names(self): return ['gnn_e0_data.pt']

    def download(self): pass

    def process(self):
        print(f'Building graph dataset ({len(self._df):,} sequences)...')
        data_list = [row_to_graph(row, self._ss) for _, row in self._df.iterrows()]
        data, slices = self.collate(data_list)
        torch.save((data, slices), self.processed_paths[0])
        print(f'Saved {len(data_list)} graphs → {self.processed_paths[0]}')


os.makedirs(PROCESSED_DIR, exist_ok=True)

train_root = os.path.join(PROCESSED_DIR, 'train')
val_root   = os.path.join(PROCESSED_DIR, 'val')
test_root  = os.path.join(PROCESSED_DIR, 'test')

# Delete cached files if re-running to force rebuild
for rdir in [train_root, val_root, test_root]:
    pdir = os.path.join(rdir, 'processed')
    if os.path.isdir(pdir):
        for fn in os.listdir(pdir):
            if fn == 'gnn_e0_data.pt':
                os.remove(os.path.join(pdir, fn))

train_gds = NNNGraphDataset(train_df, sumstats, train_root)
val_gds   = NNNGraphDataset(val_df,   sumstats, val_root)
test_gds  = NNNGraphDataset(test_df,  sumstats, test_root)

train_loader = PyGLoader(train_gds, batch_size=config['batch_size'], shuffle=True)
val_loader   = PyGLoader(val_gds,   batch_size=512,                  shuffle=False)
test_loader  = PyGLoader(test_gds,  batch_size=512,                  shuffle=False)

print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

Building graph dataset (25,025 sequences)...


Processing...


Saved 25025 graphs → data\models\processed_v2\train\processed\gnn_e0_data.pt
Building graph dataset (1,318 sequences)...


Done!
Processing...


Saved 1318 graphs → data\models\processed_v2\val\processed\gnn_e0_data.pt
Building graph dataset (1,387 sequences)...


Done!
Processing...


Saved 1387 graphs → data\models\processed_v2\test\processed\gnn_e0_data.pt
Train batches: 98  Val batches: 3


Done!


In [6]:
# ── 6. Model: GNN (TransformerConv + Set2Set) ─────────────────────────────────

class GNN(nn.Module):
    """
    TransformerConv GNN with Set2Set pooling for graph-level regression.
    
    Architecture:
      - 1 input TransformerConv (4 → hidden, heads=1)
      - 3 deeper TransformerConv (hidden → hidden, heads=4, concat=False)
      - Set2Set pooling (LSTM-based, permutation-invariant) → (2*hidden,)
      - 3-layer MLP → 2 outputs [dH_norm, Tm_norm]
    """
    def __init__(self, hidden=125, n_conv=4, lin_hidden=128,
                 conv_drop=0.0127, lin_drop=0.25, s2s_steps=10):
        super().__init__()
        self.conv_drop = conv_drop; self.lin_drop = lin_drop
        self.convs = ModuleList()
        self.convs.append(TransformerConv(4, hidden, heads=1, edge_dim=3, dropout=conv_drop))
        for _ in range(n_conv-1):
            self.convs.append(TransformerConv(hidden, hidden, heads=4, edge_dim=3,
                                              dropout=conv_drop, concat=False))
        self.pool = Set2Set(hidden, processing_steps=s2s_steps)
        self.lin1 = Linear(2*hidden, lin_hidden)
        self.lin2 = Linear(lin_hidden, lin_hidden)
        self.lin3 = Linear(lin_hidden, 2)

    def forward(self, x, edge_index, edge_attr, batch):
        for conv in self.convs:
            x = F.leaky_relu(conv(x, edge_index, edge_attr))
            x = F.dropout(x, p=self.conv_drop, training=self.training)
        x = self.pool(x, batch)                           # (B, 2*hidden)
        x = F.dropout(F.relu(self.lin1(x)), p=self.lin_drop, training=self.training)
        x = F.relu(self.lin2(x))
        return self.lin3(x)                               # (B, 2)


model = GNN(
    hidden=config['hidden_channels'], n_conv=config['n_conv_layers'],
    lin_hidden=config['linear_hidden'], conv_drop=config['conv_dropout'],
    lin_drop=config['linear_dropout'], s2s_steps=config['set2set_steps']
).to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: GNN (TransformerConv+Set2Set)  |  Parameters: {n_params:,}')

Model: GNN (TransformerConv+Set2Set)  |  Parameters: 859,023


In [7]:
# ── 7. Metrics & Evaluation Helpers ──────────────────────────────────────────

def compute_metrics(pred_norm, true_norm, sumstats):
    if torch.is_tensor(pred_norm): pred_norm = pred_norm.cpu().numpy()
    if torch.is_tensor(true_norm): true_norm = true_norm.cpu().numpy()
    dH_p = pred_norm[:,0]*(sumstats['dH_max']-sumstats['dH_min'])+sumstats['dH_min']
    Tm_p = pred_norm[:,1]*(sumstats['Tm_max']-sumstats['Tm_min'])+sumstats['Tm_min']
    dH_t = true_norm[:,0]*(sumstats['dH_max']-sumstats['dH_min'])+sumstats['dH_min']
    Tm_t = true_norm[:,1]*(sumstats['Tm_max']-sumstats['Tm_min'])+sumstats['Tm_min']
    dG_p = dH_p*(1.-(273.15+37.)/(273.15+Tm_p))
    dG_t = dH_t*(1.-(273.15+37.)/(273.15+Tm_t))
    metrics = {}
    for tag,p,t in [('dH',dH_p,dH_t),('Tm',Tm_p,Tm_t),('dG_37',dG_p,dG_t)]:
        mask = np.isfinite(t)&np.isfinite(p)
        if mask.sum()<2:
            metrics[f'{tag}_mae']=metrics[f'{tag}_rmse']=metrics[f'{tag}_r2']=float('nan')
        else:
            d=p[mask]-t[mask]
            metrics[f'{tag}_mae']  = float(np.mean(np.abs(d)))
            metrics[f'{tag}_rmse'] = float(np.sqrt(np.mean(d**2)))
            metrics[f'{tag}_r2']   = float(r2_score(t[mask],p[mask]))
    return metrics, dH_p, Tm_p, dH_t, Tm_t


@torch.no_grad()
def evaluate(model, loader, sumstats, device):
    model.eval(); preds, trues = [], []
    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)  # (B, 2)
        preds.append(out.cpu()); trues.append(batch.y.view(-1,2).cpu())
    return compute_metrics(torch.cat(preds), torch.cat(trues), sumstats)

print('Metrics helpers defined.')

Metrics helpers defined.


In [8]:
# ── 8. Training Loop ──────────────────────────────────────────────────────────

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=config['lr'], weight_decay=config['weight_decay'])
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config['n_epoch'], eta_min=1e-5)

history = {'train_loss':[], 'val_dH_mae':[], 'val_Tm_mae':[], 'val_dG_mae':[], 'val_dH_rmse':[], 'val_Tm_rmse':[]}
os.makedirs(config['checkpoint_dir'], exist_ok=True)
os.makedirs('out', exist_ok=True)

_run_name = 'E0_GNN_TransformerConv_Set2Set'
_kw = dict(project=config['wandb_project'], name=_run_name, config=config, reinit=True)
_mode = os.environ.get('WANDB_MODE','').strip().lower()
if _mode in ('offline','disabled'):
    run = wandb.init(mode=_mode, **_kw)
else:
    try:    run = wandb.init(**_kw)
    except Exception as e:
        print(f'WandB online failed ({e}), offline.'); run = wandb.init(mode='offline', **_kw)
print(f'WandB run: {run.name}  |  mode: {run.settings.mode}')

best_val_dG = float('inf'); start = time.time()

for epoch in range(config['n_epoch']):
    model.train(); train_loss = 0.
    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)  # (B,2)
        y    = batch.y.view(-1,2)                                               # (B,2)
        loss = criterion(out, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), config['grad_clip'])
        optimizer.step()
        train_loss += loss.item() * batch.num_graphs
    train_loss /= len(train_loader.dataset)
    scheduler.step()

    vm, *_ = evaluate(model, val_loader, sumstats, device)
    history['train_loss'].append(train_loss)
    history['val_dH_mae'].append(vm['dH_mae'])
    history['val_Tm_mae'].append(vm['Tm_mae'])
    history['val_dG_mae'].append(vm['dG_37_mae'])
    history['val_dH_rmse'].append(vm['dH_rmse'])
    history['val_Tm_rmse'].append(vm['Tm_rmse'])

    wandb.log({'epoch':epoch,'train_loss':train_loss,**{f'val_{k}':v for k,v in vm.items()},'lr':scheduler.get_last_lr()[0]})

    if vm['dG_37_mae'] < best_val_dG:
        best_val_dG = vm['dG_37_mae']
        torch.save(model.state_dict(), os.path.join(config['checkpoint_dir'],'best_gnn_model.pt'))

    if (epoch+1) % 20 == 0:
        print(f"Ep {epoch+1:3d}/{config['n_epoch']} | loss {train_loss:.4f} | dH {vm['dH_mae']:.3f} | Tm {vm['Tm_mae']:.3f} | dG {vm['dG_37_mae']:.3f} | {(time.time()-start)/60:.1f}min")

run.finish()
with open('out/gnn_history.json','w') as f: json.dump(history, f)
print(f'Done. Best val dG MAE: {best_val_dG:.4f}')

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from C:\Users\anant\.netrc.


wandb: Currently logged in as: apati087 (apati087-university-of-california-riverside) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


WandB run: E0_GNN_TransformerConv_Set2Set  |  mode: online
Ep  20/200 | loss 0.0072 | dH 3.935 | Tm 3.612 | dG 0.315 | 2.5min
Ep  40/200 | loss 0.0058 | dH 3.694 | Tm 4.243 | dG 0.370 | 4.9min
Ep  60/200 | loss 0.0054 | dH 3.553 | Tm 3.396 | dG 0.304 | 7.3min
Ep  80/200 | loss 0.0049 | dH 3.468 | Tm 3.901 | dG 0.325 | 9.7min
Ep 100/200 | loss 0.0045 | dH 3.414 | Tm 3.978 | dG 0.346 | 12.1min
Ep 120/200 | loss 0.0042 | dH 3.331 | Tm 4.438 | dG 0.376 | 14.5min
Ep 140/200 | loss 0.0039 | dH 3.266 | Tm 4.309 | dG 0.361 | 16.9min
Ep 160/200 | loss 0.0038 | dH 3.211 | Tm 4.217 | dG 0.357 | 19.3min
Ep 180/200 | loss 0.0036 | dH 3.160 | Tm 4.148 | dG 0.357 | 21.7min
Ep 200/200 | loss 0.0036 | dH 3.159 | Tm 4.260 | dG 0.366 | 24.2min


epoch,▁▁▁▃▃▃▃▄▄▄▄▄▄▄▄▄▄▄▅▅▆▆▆▆▆▆▆▆▆▇▇▇▇▇▇▇████
lr,███████▇▇▇▇▆▆▆▆▅▅▅▅▅▄▄▄▃▃▃▃▃▃▂▂▂▁▁▁▁▁▁▁▁
train_loss,█▆▆▅▅▄▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val_Tm_mae,█▇▅▄▃▂▂▃▁▂▃▁▁▁▃▃▃▂▃▃▂▄▂▃▄▄▃▃▃▄▃▃▃▃▃▃▃▃▃▃
val_Tm_r2,▁▅▆▇▆▆█▆▇▇█▇▇▆▇▇▇▇▇▇▇██▇▇▆▇▇▇▇▆▇▇▆▇▇▇▇▇▇
val_Tm_rmse,█▂▂▁▂▁▂▂▁▁▂▁▁▂▂▂▂▂▂▁▂▂▂▂▁▂▂▂▂▂▂▂▂▂▁▂▂▂▂▂
val_dG_37_mae,▇▄▃▄▆▁▄▄▃█▆▆▆▃▂▄▆▄▄▆▅▇▇▇▅▆▆▅▆▅▆▅▅▆▆▆▅▅▆▆
val_dG_37_r2,▁▆█████▇▇█▇████▇▇██▇██▇█▇▇█▇▇▇██▇█▇▇▇▇▇▇
val_dG_37_rmse,█▃▂▂▂▂▁▁▂▁▂▂▂▂▁▂▂▁▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂
val_dH_mae,█▄▃▃▃▂▂▂▂▂▂▁▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
+2,...


Done. Best val dG MAE: 0.2659


In [13]:
# ── 9. Final Evaluation ───────────────────────────────────────────────────────

model.load_state_dict(torch.load(os.path.join(config['checkpoint_dir'],'best_gnn_model.pt'), map_location=device))
val_m,  dH_vp, Tm_vp, dH_vt, Tm_vt = evaluate(model, val_loader,  sumstats, device)
test_m, dH_tp, Tm_tp, dH_tt, Tm_tt = evaluate(model, test_loader, sumstats, device)

print('=== Val (arr) ===');  [print(f'  {t}  MAE {val_m[f"{k}_mae"]:.3f}  R2 {val_m[f"{k}_r2"]:.3f}') for t,k in [('dH','dH'),('Tm','Tm'),('dG37','dG_37')]]
print('=== Test (arr) ==='); [print(f'  {t}  MAE {test_m[f"{k}_mae"]:.3f}  R2 {test_m[f"{k}_r2"]:.3f}') for t,k in [('dH','dH'),('Tm','Tm'),('dG37','dG_37')]]

ev = pd.DataFrame({'dH_pred':dH_vp,'dH_true':dH_vt,'Tm_pred':Tm_vp,'Tm_true':Tm_vt})
ev['dG_pred'] = ev['dH_pred']*(1-310.15/(273.15+ev['Tm_pred']))
ev['dG_true'] = ev['dH_true']*(1-310.15/(273.15+ev['Tm_true']))
ev.to_csv('out/gnn_val_eval.csv', index=False)

run_log = dict(experiment_id='E0', model='GNN_TransformerConv_Set2Set', config=config,
               n_params=sum(p.numel() for p in model.parameters() if p.requires_grad),
               val_metrics=val_m, test_metrics=test_m,
               best_checkpoint=os.path.join(config['checkpoint_dir'],'best_gnn_model.pt'))
with open('out/gnn_run_log.json','w') as f: json.dump(run_log, f, indent=2)
print('Saved: out/gnn_val_eval.csv  out/gnn_run_log.json')

=== Val (arr) ===
  dH  MAE 3.682  R2 0.801
  Tm  MAE 3.283  R2 0.840
  dG37  MAE 0.266  R2 0.857
=== Test (arr) ===
  dH  MAE 3.703  R2 0.810
  Tm  MAE 3.189  R2 0.842
  dG37  MAE 0.262  R2 0.864
Saved: out/gnn_val_eval.csv  out/gnn_run_log.json


In [14]:
# ── 10. Convergence Curves (F2 contribution) ──────────────────────────────────
os.makedirs('out/figures', exist_ok=True)
ep = range(1, len(history['train_loss'])+1)
fig,axes = plt.subplots(1,3,figsize=(14,4),facecolor='#f8f9fa')
for ax,(k,yl) in zip(axes,[('val_dH_mae','Val dH MAE (kcal/mol)'),('val_Tm_mae','Val Tm MAE (deg C)'),('val_dG_mae','Val dG37 MAE (kcal/mol)')]):
    ax.plot(ep,history[k],color=MODEL_COLOR,lw=2,label='E0: GNN')
    ax.set_xlabel('Epoch'); ax.set_ylabel(yl); ax.legend(fontsize=9); sns.despine(ax=ax)
fig.suptitle('E0: GNN (TransformerConv + Set2Set) Convergence',fontsize=11,fontweight='bold')
plt.tight_layout()
plt.savefig('out/figures/gnn_convergence.png',dpi=300,bbox_inches='tight')
plt.show(); print('Saved: out/figures/gnn_convergence.png')

Saved: out/figures/gnn_convergence.png


C:\Users\anant\AppData\Local\Temp\ipykernel_37652\3031337454.py:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show(); print('Saved: out/figures/gnn_convergence.png')


In [15]:
# ── 11. Scatter Plots (F3 contribution) ───────────────────────────────────────
AXIS_LIMITS = {'dH':(-55,-5),'Tm':(20,60),'dG_37':(-7,5)}
dG_vp = dH_vp*(1-310.15/(273.15+Tm_vp)); dG_vt = dH_vt*(1-310.15/(273.15+Tm_vt))
fig,axes = plt.subplots(1,3,figsize=(14,5),facecolor='#f8f9fa')
for ax,(p,t,tag,unit) in zip(axes,[(dH_vp,dH_vt,'dH','kcal/mol'),(Tm_vp,Tm_vt,'Tm','deg C'),(dG_vp,dG_vt,'dG_37','kcal/mol')]):
    lim=AXIS_LIMITS[tag]; m=np.isfinite(p)&np.isfinite(t)
    ax.scatter(t[m],p[m],s=4,alpha=0.4,color=MODEL_COLOR,rasterized=True)
    ax.plot(lim,lim,'k--',alpha=0.3,lw=1.5)
    mae=np.mean(np.abs(p[m]-t[m])); r2=r2_score(t[m],p[m])
    ax.text(0.05,0.93,f'MAE={mae:.3f}\nR2={r2:.3f}',transform=ax.transAxes,fontsize=8.5,va='top',bbox=dict(boxstyle='round,pad=0.3',fc='white',alpha=0.8))
    ax.set_xlim(lim); ax.set_ylim(lim); ax.set_xlabel(f'Measured {tag}'); ax.set_ylabel(f'Predicted {tag}'); ax.set_title(tag,fontweight='bold'); sns.despine(ax=ax)
fig.suptitle('E0: GNN Predicted vs Measured (Val)',fontsize=11)
plt.tight_layout()
plt.savefig('out/figures/gnn_scatter.png',dpi=300,bbox_inches='tight')
plt.show(); print('Saved: out/figures/gnn_scatter.png')

Saved: out/figures/gnn_scatter.png


C:\Users\anant\AppData\Local\Temp\ipykernel_37652\4288428771.py:15: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  plt.show(); print('Saved: out/figures/gnn_scatter.png')


In [16]:
# ── 12. lit_uv Generalization (F4 contribution) ───────────────────────────────
lit_df = df[df['dataset']=='lit_uv'].copy()
lit_df = lit_df.dropna(subset=['Tm'])
print(f'lit_uv: {len(lit_df)} sequences (Tm-only)')

# Build temporary InMemoryDataset for lit_uv
lit_root = os.path.join(PROCESSED_DIR, 'lit_uv')
pdir = os.path.join(lit_root, 'processed')
if os.path.isdir(pdir):
    for fn in os.listdir(pdir):
        if fn == 'gnn_e0_data.pt': os.remove(os.path.join(pdir, fn))

# lit_uv has Tm but not dH — fill dH with train mean for normalization purposes only
lit_df = lit_df.copy()
if lit_df['dH'].isna().all():
    lit_df['dH'] = train_df['dH'].mean()

lit_gds    = NNNGraphDataset(lit_df, sumstats, lit_root)
lit_loader = PyGLoader(lit_gds, batch_size=128, shuffle=False)

model.eval(); Tm_preds = []
with torch.no_grad():
    for batch in lit_loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr, batch.batch)  # (B,2)
        Tm_preds.extend(unnormalize(out[:,1].cpu().numpy(), sumstats['Tm_min'], sumstats['Tm_max']))

Tm_true = lit_df['Tm'].values
lit_mae = float(np.mean(np.abs(np.array(Tm_preds)-Tm_true)))
print(f'lit_uv Tm MAE: {lit_mae:.3f} degC')

with open('out/gnn_run_log.json') as f: rl = json.load(f)
rl['lit_uv_Tm_mae'] = lit_mae
with open('out/gnn_run_log.json','w') as f: json.dump(rl, f, indent=2)
print('Updated gnn_run_log.json with lit_uv result.')

Processing...
Done!


lit_uv: 348 sequences (Tm-only)
Building graph dataset (348 sequences)...
Saved 348 graphs → data\models\processed_v2\lit_uv\processed\gnn_e0_data.pt
lit_uv Tm MAE: 5.508 degC
Updated gnn_run_log.json with lit_uv result.
